In [ ]:
import os
import time
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.ops import deform_conv2d
import timm
from tqdm import tqdm

# ==========================================
#               CONFIGURATION
# ==========================================
CONFIG = {
    'model_name': 'resnet18', # Changed to ResNet18
    'img_size': 224,
    'batch_size': 64, # ResNet18 is lighter, can handle larger batch size
    'epochs': 15,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'patience': 5,
    'num_classes': 100, # CIFAR-100
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'data_dir': 'CIFAR100',  # <--- Point this to your extracted DIRECTORY
    'checkpoint_path': 'hybrid_resnet18_dcnn_cifar100_best.pth',
    'seed': 42,
}

# ==========================================
#           1. UTILITIES & DATA
# ==========================================
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

def get_dataloaders(data_dir):
    cifar100_mean = [0.5071, 0.4867, 0.4408]
    cifar100_std = [0.2675, 0.2565, 0.2761]

    transform_train = transforms.Compose([
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])), 
        transforms.RandomCrop(CONFIG['img_size'], padding=16),       
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=cifar100_mean, std=cifar100_std)
    ])

    transform_val = transforms.Compose([
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=cifar100_mean, std=cifar100_std)
    ])

    # --- ROBUST LOADING LOGIC ---
    abs_data_dir = os.path.abspath(data_dir)
    
    if os.path.isfile(abs_data_dir):
        raise ValueError(f"ERROR: '{abs_data_dir}' is a FILE, not a directory. Please extract your dataset.")
    if not os.path.isdir(abs_data_dir):
        raise ValueError(f"ERROR: Directory '{abs_data_dir}' does not exist.")

    print(f"Checking contents of: {abs_data_dir}")
    print(f"Found items: {os.listdir(abs_data_dir)}")

    official_cifar_dir = os.path.join(abs_data_dir, 'cifar-100-python')
    if os.path.isdir(official_cifar_dir):
        print("✅ Found official PyTorch CIFAR-100 format.")
        train_dataset = datasets.CIFAR100(root=abs_data_dir, train=True, download=False, transform=transform_train)
        val_dataset = datasets.CIFAR100(root=abs_data_dir, train=False, download=False, transform=transform_val)
    else:
        train_img_dir = os.path.join(abs_data_dir, 'train')
        test_img_dir = os.path.join(abs_data_dir, 'test')
        val_img_dir = os.path.join(abs_data_dir, 'val')

        if os.path.isdir(train_img_dir):
            print("✅ Found extracted image folders. Using ImageFolder.")
            train_dataset = datasets.ImageFolder(root=train_img_dir, transform=transform_train)
            val_dir = test_img_dir if os.path.isdir(test_img_dir) else val_img_dir
            if not os.path.isdir(val_dir):
                raise ValueError(f"Found 'train' folder, but could not find 'test' or 'val' folder.")
            val_dataset = datasets.ImageFolder(root=val_dir, transform=transform_val)
        else:
            raise ValueError(f"❌ ERROR: Could not find a valid dataset structure in '{abs_data_dir}'.")

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Total Classes Found: {len(train_dataset.classes)}")
    print(f"Train Images: {len(train_dataset)} | Val Images: {len(val_dataset)}")
    
    return train_loader, val_loader

# ==========================================
#        2. ARCHITECTURE COMPONENTS
# ==========================================

class LocalCNNBranch(nn.Module):
    """
    Downsamples 224x224 -> 7x7 to match the ResNet18 layer4 feature map.
    """
    def __init__(self, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            # 224 -> 112
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            # 112 -> 56
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            # 56 -> 28
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(),
            # 28 -> 14
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(),
            # 14 -> 7 (Matches ResNet18 layer4 spatial dimensions)
            nn.Conv2d(512, out_dim, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_dim), nn.SiLU()
        )

    def forward(self, x):
        return self.net(x)

class CrossAttentionFusion(nn.Module):
    """
    Fuses Local CNN features (Query) with Global ResNet features (Key/Value).
    """
    def __init__(self, dim, num_heads=4, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5

        self.q_proj = nn.Linear(dim, dim, bias=qkv_bias) 
        self.k_proj = nn.Linear(dim, dim, bias=qkv_bias) 
        self.v_proj = nn.Linear(dim, dim, bias=qkv_bias) 

        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x_local, x_global):
        B, C, H, W = x_local.shape
        x_local_flat = x_local.flatten(2).transpose(1, 2) 
        B_v, N_vit, C_v = x_global.shape

        q = self.q_proj(x_local_flat).reshape(B, H*W, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = self.k_proj(x_global).reshape(B, N_vit, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = self.v_proj(x_global).reshape(B, N_vit, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, H*W, C)
        x = self.proj(x)
        x = self.norm(x + x_local_flat)
        
        return x.transpose(1, 2).reshape(B, C, H, W)

class DeformableDCNNHead(nn.Module):
    """
    Applies Deformable Convolution to handle irregular boundaries.
    """
    def __init__(self, dim, num_classes):
        super().__init__()
        self.offset_conv = nn.Conv2d(dim, 2 * 3 * 3, kernel_size=3, padding=1)
        self.deform_weight = nn.Parameter(torch.Tensor(dim, dim, 3, 3))
        nn.init.kaiming_uniform_(self.deform_weight, nonlinearity='relu')
        
        self.bn = nn.BatchNorm2d(dim)
        self.act = nn.GELU()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(dim, num_classes)

    def forward(self, x):
        offsets = self.offset_conv(x)
        x = deform_conv2d(x, offsets, self.deform_weight, padding=1)
        x = self.act(self.bn(x))
        x = self.global_pool(x).flatten(1)
        logits = self.classifier(x)
        return logits

# ==========================================
#           3. THE HYBRID MODEL
# ==========================================
class HybridResNet18_DCNN(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        
        # --- Branch A: ResNet18 Backbone ---
        # features_only=True returns spatial feature maps. 
        # out_indices=(4,) grabs the final stage (layer4) before global pooling.
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True, out_indices=(4,))
        self.embed_dim = self.backbone.feature_info.channels()[-1] # 512 for ResNet18
        
        # --- Branch B: Local CNN ---
        self.cnn = LocalCNNBranch(out_dim=self.embed_dim)
        
        # --- Fusion: Cross Attention ---
        self.fusion = CrossAttentionFusion(dim=self.embed_dim)
        
        # --- Head: Deformable CNN ---
        self.head = DeformableDCNNHead(dim=self.embed_dim, num_classes=num_classes)

    def forward(self, x):
        # 1. Global Features (ResNet18)
        # feats is a list, we take the last element: [Batch, 512, 7, 7]
        feats = self.backbone(x)
        x_resnet = feats[-1] 
        
        # Flatten spatial dimensions to match CrossAttention expected shape [Batch, N, C]
        B, C, H, W = x_resnet.shape
        x_resnet_flat = x_resnet.flatten(2).transpose(1, 2) # [Batch, 49, 512]
        
        # 2. Local Features (CNN)
        # Result: [Batch, 512, 7, 7]
        x_cnn = self.cnn(x) 
        
        # 3. Cross Attention Fusion
        x_fused = self.fusion(x_local=x_cnn, x_global=x_resnet_flat)
        
        # 4. Deformable Classification
        logits = self.head(x_fused)
        
        return logits

# ==========================================
#             4. TRAINING LOOP
# ==========================================
def train_engine():
    seed_everything(CONFIG['seed'])
    
    print("Initializing Data Loaders...")
    train_loader, val_loader = get_dataloaders(CONFIG['data_dir'])
    
    print(f"Initializing Hybrid Model ({CONFIG['model_name']})...")
    model = HybridResNet18_DCNN(CONFIG['model_name'], CONFIG['num_classes']).to(CONFIG['device'])
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    
    best_acc = 0.0
    
    print("Starting Training...")
    for epoch in range(CONFIG['epochs']):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Train]")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(CONFIG['device']), labels.to(CONFIG['device'])
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'loss': running_loss/total, 'acc': correct/total})
            
        train_acc = correct / total
        
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(CONFIG['device']), labels.to(CONFIG['device'])
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        val_acc = val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1} Result: Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        scheduler.step(val_acc)
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), CONFIG['checkpoint_path'])
            print(f"--> New Best Model Saved! ({best_acc:.4f})")
            
    print(f"\nTraining Complete. Best Validation Accuracy: {best_acc:.4f}")

if __name__ == "__main__":
    train_engine()